# Pumpekraftverket

## Hydrauliske nettverk, svingesjakt og dynamisk effektregulering

### Pilotprosjekt for Matematikk 1, VVS og fornybar energi

Et pumpekraftverk har et øvre magasin, en lang tilløpstunnel, en svingesjakt, et trykkrør og en reversibel pumpe-turbin. I turbinedrift strømmer vannet fra øvre til nedre magasin og produserer elektrisk energi. I pumpedrift brukes elektrisk energi til å flytte vann tilbake til øvre magasin.

Prosjektet følger flere modellnivåer:

1. **Stasjonært hydraulisk nettverk:** trykkhøyder og vannføringer fra $B^TGB$
2. **Én stiv vannkolonne:** skalar ODE, likevekt og tidskonstant
3. **Svingesjakt:** koblet vektor-ODE for tunnelstrømning, trykkrørstrømning og nivå
4. **Ledeskovl:** firetilstandsmodell og effektregulering
5. **Egenmoder:** linearisert modell, demping og oscillasjonsfrekvens

### Læringsmål

Etter prosjektet skal du kunne

- bygge et hydraulisk nettverk fra en forbindelsesmatrise,
- løse trykkhøyder og vannføringer med et lineært system,
- kontrollere kontinuitet og hydraulisk effekt,
- utlede en skalar vannkolonne-ODE fra momentbalanse,
- finne likevekt og tidskonstant,
- implementere lineære og ikke-lineære tap,
- skrive svingesjaktmodellen som et vektor-ODE-system,
- bruke Euler på et firetilstandssystem,
- finne og tolke en likevekt,
- beregne egenverdier og egenvektorer til en linearisert modell,
- tolke realdel, imaginærdel og hydrauliske moder,
- forklare overgangen fra en samlet ODE-modell til en PDE-modell for vannhammer.

### Modellavgrensning

Hovedmodellen antar stive rør og inkompressibelt vann. Den beskriver langsomme vannmasse- og svingesjaktbevegelser, ikke vandrende trykkbølger. Turbin- og regulatormodellene er pedagogiske og skal ikke brukes til dimensjonering eller sikkerhetsanalyse av virkelige anlegg.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rho = 1000.0  # kg/m^3
g = 9.81      # m/s^2

# Referanseanlegget

Parameterne er inspirert av et eksempelanlegg i den faglige bakgrunnen, men avrundet og tilpasset undervisning.

## Magasin og vannvei

- øvre magasin: $307.0$ m over valgt datum
- nedre magasin: $166.4$ m
- statisk fallhøyde: $140.6$ m
- tilløpstunnel: lengde $3850$ m, areal $38.5$ m²
- trykkrør: lengde $465$ m, areal $15.2$ m²
- svingesjakt: areal $78.5$ m²
- nominell vannføring: omtrent $71.4$ m³/s

## Turbin og ledeskovl

- nominell ledeskovlåpning: $0.90$
- servotidskonstant: $0.50$ s
- turbinvirkningsgrad: $0.90$

Disse verdiene gir et anlegg i størrelsesorden 80 til 90 MW i referansetilstanden.

In [ ]:
H_øvre_abs = 307.0
H_nedre_abs = 166.4
H_statisk = H_øvre_abs - H_nedre_abs

L_tunnel = 3850.0
A_tunnel = 38.5
L_trykkrør = 465.0
A_trykkrør = 15.2
A_sjakt = 78.5

Q_ref = 71.43
y_ref_normal = 0.90
T_servo = 0.50
eta_turbin = 0.90

# Vannkolonnens treghetskoeffisienter i den valgte modellen.
M_tunnel = L_tunnel/(g*A_tunnel)
M_trykkrør = L_trykkrør/(g*A_trykkrør)

# Kvadratiske tapskoeffisienter, h_tap = R*Q*abs(Q).
R_tunnel = 0.00101
R_trykkrør = 0.0003042

print("Statisk fallhøyde:", H_statisk, "m")
print("M_tunnel:", M_tunnel)
print("M_trykkrør:", M_trykkrør)

# Del A: Stasjonært hydraulisk nettverk

## A.1 To reversible enheter i parallell

Det stasjonære nettverket brukes for å introdusere lineær algebra. To aggregater forsynes fra en felles tunnel og hvert sitt korte trykkrør.

```text
øvre magasin -- tunnel -- fordelingspunkt -- trykkrør 1 -- aggregat 1 -- nedre magasin
                               |
                               +----------- trykkrør 2 -- aggregat 2 -- nedre magasin
```

Nodene er:

0. øvre magasin, kjent høyde
1. fordelingspunkt
2. turbininntak 1
3. turbininntak 2
4. nedre magasin, kjent høyde

Kantene er tunnel, to trykkrør og to aggregatgrener.

Vi bruker den lineariserte sammenhengen

$$
\boxed{q_e=g_e(h_i-h_j).}
$$

Dette er en lokal nettverksmodell, ikke den kvadratiske tapsloven som brukes senere.

In [ ]:
node_navn = ["øvre magasin", "fordelingspunkt", "inntak 1", "inntak 2", "nedre magasin"]
kant_navn = ["tunnel", "trykkrør 1", "trykkrør 2", "aggregat 1", "aggregat 2"]

# Konduktanser valgt rundt en referanse med omtrent 35 m^3/s per aggregat.
g_lin = np.array([14.0, 23.3, 23.3, 0.261, 0.261])

B = np.array([
    [ 1.0, -1.0,  0.0,  0.0,  0.0],
    [ 0.0,  1.0, -1.0,  0.0,  0.0],
    [ 0.0,  1.0,  0.0, -1.0,  0.0],
    [ 0.0,  0.0,  1.0,  0.0, -1.0],
    [ 0.0,  0.0,  0.0,  1.0, -1.0]
])

## Oppgave A1: Bygg nettverksmatrisen

Definer

$$G=\operatorname{diag}(g_1,\ldots,g_5),$$

$$
\boxed{K=B^TGB.}
$$

Kontroller symmetri, nullrom og egenverdier.

In [ ]:
G_lin = ...
K_full = ...

print("K =
", K_full)
print("Symmetrisk:", ...)
print("K @ 1 =", ...)
print("Egenverdier:", ...)

## A.2 Fest magasinshøydene

Høydene i øvre og nedre magasin er kjente. Ukjente noder er 1, 2 og 3.

Del systemet i ukjente og kjente variable:

$$
\boxed{K_{uu}h_u=-K_{ur}h_r.}
$$

In [ ]:
ukjente = np.array([1, 2, 3])
rand = np.array([0, 4])

h_rand = np.array([H_statisk, 0.0])  # relativt til nedre magasin
K_uu = K_full[np.ix_(ukjente, ukjente)]
K_ur = K_full[np.ix_(ukjente, rand)]

h_u = ...
h_stasjonær = np.zeros(5)
h_stasjonær[rand] = h_rand
h_stasjonær[ukjente] = h_u

print("Nodehøyder:", h_stasjonær)
print("Residual:", ...)

## Oppgave A2: Vannføring og kontinuitet

Beregn

$$q=GBh.$$

Kontroller at kontinuiteten holder i de tre indre nodene.

In [ ]:
q_kant = ...
nodebalanse = ...

for navn, q in zip(kant_navn, q_kant):
    print(f"{navn:12s}: {q:7.3f} m^3/s")
print("Indre nodebalanser:", nodebalanse[ukjente])

## Oppgave A3: Effekt og drift med ett aggregat

For hvert aggregat bruker vi

$$
P=\eta\rho gQH.
$$

1. Beregn total turbineffekt.
2. Sett konduktansen til aggregat 2 lik null og løs nettet på nytt.
3. Sammenlign vannføring, fallhøyde og effekt.
4. Diskuter hvorfor den lineære modellen bare er rimelig nær referansedriften.

In [ ]:
Q_agg = q_kant[3:5]
H_agg = h_stasjonær[2:4] - h_stasjonær[4]
P_agg = ...
print("Effekt per aggregat, MW:", P_agg/1e6)
print("Total effekt, MW:", np.sum(P_agg)/1e6)

# Del B: Én stiv vannkolonne

## B.1 Momentbalanse

For en rørledning med samlet vannføring $Q(t)$ bruker vi

$$
\boxed{
M_Q\dot Q
=
\Delta H-RQ,
\qquad
M_Q=\frac{L}{gA}.}
$$

Den lineære tapsmodellen egner seg for håndløsning og små avvik rundt et driftspunkt.

In [ ]:
M_Q = M_tunnel + M_trykkrør
Delta_H = H_statisk

# Velg lineær motstand slik at Q_ref er likevekt.
R_lineær = Delta_H/Q_ref

## Oppgave B1: Likevekt og tidskonstant

Finn

$$Q^*=\frac{\Delta H}{R},$$

$$\tau=\frac{M_Q}{R}.$$

Løs analytisk for $Q(0)=0$.

In [ ]:
Q_likevekt = ...
tau_Q = ...

print("Likevekt:", Q_likevekt)
print("Tidskonstant:", tau_Q, "s")


def Q_analytisk(t, Q0=0.0):
    return ...

## Oppgave B2: Euler mot analytisk løsning

Sammenlign flere tidssteg. Hva skjer dersom fallhøyden skifter fortegn, slik at anlegget går over til pumpedrift i den idealiserte modellen?

In [ ]:
def euler_skalar(f, y0, sluttid, dt):
    n = int(round(sluttid/dt))
    t = np.linspace(0.0, n*dt, n+1)
    y = np.zeros(n+1)
    y[0] = y0
    for k in range(n):
        y[k+1] = ...
    return t, y


def vannkolonne_lineær(t, Q):
    return (Delta_H-R_lineær*Q)/M_Q

# Kjør og plott mot Q_analytisk.

## B.2 Ikke-lineært tap

En mer fysisk tapsmodell er

$$
\boxed{M_Q\dot Q=\Delta H-R_qQ|Q|.}
$$

Velg $R_q$ slik at $Q_{ref}$ er likevekt.

In [ ]:
R_kvadrat = Delta_H/(Q_ref*abs(Q_ref))


def vannkolonne_nonlineær(t, Q):
    return ...

print("Kvadratisk tapskoeffisient:", R_kvadrat)

## Oppgave B3: Lineær og ikke-lineær modell

Sammenlign modellene ved

- liten endring rundt $Q_{ref}$,
- oppstart fra $Q=0$,
- stor reduksjon i drivhøyden,
- skifte av strømretning.

Forklar hvorfor den ikke-lineære modellen trenger $Q|Q|$ og ikke bare $Q^2$.

# Del C: Svingesjakt som vektor-ODE

## C.1 Tre tilstander

La

- $Q_t$: vannføring i tilløpstunnelen,
- $Q_p$: vannføring i trykkrøret,
- $z$: vannhøyde i svingesjakten relativt til nedre magasin.

Modellen er

$$
\boxed{
\begin{aligned}
M_t\dot Q_t
&=H_{statisk}-z-R_tQ_t|Q_t|,\\
M_p\dot Q_p
&=z-H_T(y,Q_p)-R_pQ_p|Q_p|,\\
A_s\dot z
&=Q_t-Q_p.
\end{aligned}}
$$

Kontinuitetsligningen sier at sjaktnivået stiger når mer vann kommer inn gjennom tunnelen enn det som går videre til turbinen.

## C.2 Turbinens trykkfall

Vi bruker

$$
\boxed{
H_T(y,Q_p)
=k_T\frac{Q_p|Q_p|}{y^2+\varepsilon}.}
$$

$k_T$ velges slik at $Q=Q_{ref}$ og $y=0.90$ er en likevekt.

In [ ]:
epsilon_y = 1e-4

z_ref = H_statisk - R_tunnel*Q_ref**2
H_turbin_ref = z_ref - R_trykkrør*Q_ref**2
k_turbin = H_turbin_ref*(y_ref_normal**2 + epsilon_y)/(Q_ref**2)

print("Referansenivå i svingesjakt:", z_ref, "m")
print("Referansefall over turbin:", H_turbin_ref, "m")
print("k_turbin:", k_turbin)


def H_turbin(y, Q):
    return k_turbin*Q*abs(Q)/(y*y + epsilon_y)

## Oppgave C1: Implementer tretilstandsmodellen

Hold ledeskovlen fast på $y=0.90$. Kontroller at referansetilstanden

$$x^*=(Q_{ref},Q_{ref},z_{ref})^T$$

har nesten null derivert.

In [ ]:
def svingesjakt_ode_fast_y(t, x):
    Qt, Qp, z = x
    dQt = ...
    dQp = ...
    dz = ...
    return np.array([dQt, dQp, dz])

x_ref_3 = np.array([Q_ref, Q_ref, z_ref])
print("Derivert i referansetilstand:", svingesjakt_ode_fast_y(0.0, x_ref_3))

## Oppgave C2: En liten forstyrrelse

Øk tunnelvannføringen med 5 prosent i starttilstanden, men behold $Q_p$ og $z$ på referanseverdiene. Simuler og plott:

- $Q_t$,
- $Q_p$,
- $z-z_{ref}$.

Undersøk virkningen av større og mindre svingesjaktareal.

In [ ]:
def euler_system(f, x0, sluttid, dt):
    n = int(round(sluttid/dt))
    t = np.linspace(0.0, n*dt, n+1)
    X = np.zeros((n+1, len(x0)))
    X[0] = x0
    for k in range(n):
        X[k+1] = ...
    return t, X

x0_C = x_ref_3.copy()
x0_C[0] *= 1.05

# Simuler for eksempel 600 sekunder.

# Del D: Ledeskovl og effektregulering

## D.1 Fjerde tilstand

Ledeskovlåpningen følger en referanse med førsteordens treghet:

$$
\boxed{T_y\dot y=y_{ref}(t)-y.}
$$

Tilstanden er

$$
\boxed{x=(Q_t,Q_p,z,y)^T.}
$$

In [ ]:
def y_referanse(t):
    # Lastøkning ved 100 s og lastavslag ved 350 s.
    if t < 100.0:
        return 0.90
    if t < 350.0:
        return 0.98
    return 0.65


def pumpekraft_ode(t, x):
    Qt, Qp, z, y = x
    dQt = (H_statisk-z-R_tunnel*Qt*abs(Qt))/M_tunnel
    dQp = (z-H_turbin(y, Qp)-R_trykkrør*Qp*abs(Qp))/M_trykkrør
    dz = (Qt-Qp)/A_sjakt
    dy = ...
    return np.array([dQt, dQp, dz, dy])

## Oppgave D1: Lastøkning og lastavslag

Start i referansetilstanden og simuler 700 sekunder. Plott alle fire tilstander.

Forklar:

- hvorfor $Q_p$ endrer seg før $Q_t$,
- hvorfor svingesjakten først leverer eller mottar differansen,
- hvorfor nivået oscillerer,
- hvordan rask og langsom servo påvirker responsen.

In [ ]:
x_ref_4 = np.array([Q_ref, Q_ref, z_ref, y_ref_normal])

# Simuler, plott og sammenlign ulike T_servo.

## D.2 Mekanisk effekt

Turbineffekten beregnes som

$$
\boxed{P_m=\eta\rho gQ_pH_T.}
$$

I modellen brukes fortegnet til $Q_p$ for å skille strømretningen. Ved pumpedrift må egen pumpevirkningsgrad og motorkarakteristikk brukes.

In [ ]:
def turbineffekt(Qp, y):
    return eta_turbin*rho*g*Qp*H_turbin(y, Qp)

P_ref = turbineffekt(Q_ref, y_ref_normal)
print("Referanseeffekt:", P_ref/1e6, "MW")

## Oppgave D2: Energi og driftsgrenser

Beregn effekt gjennom simuleringen. Registrer

- maksimal og minimal vannføring,
- maksimal og minimal svingesjakthøyde,
- maksimal effekt,
- tiden til ny tilnærmet likevekt.

Diskuter hvilke begrensninger en virkelig regulator må ta hensyn til.

# Del E: Linearisering og egenmoder

## E.1 Lineær tretilstandsmodell

For små avvik rundt referansetilstanden skriver vi

$$
q_t=Q_t-Q_{ref},\qquad q_p=Q_p-Q_{ref},\qquad \zeta=z-z_{ref}.
$$

En lineær modell er

$$
\boxed{
\frac{d}{dt}
\begin{pmatrix}q_t\\q_p\\\zeta\end{pmatrix}
=A_{lin}
\begin{pmatrix}q_t\\q_p\\\zeta\end{pmatrix}.}
$$

Deriverte av $RQ|Q|$ ved positiv $Q_{ref}$ gir $2RQ_{ref}$.

In [ ]:
r_t_lin = 2*R_tunnel*Q_ref
r_p_lin = 2*R_trykkrør*Q_ref
r_turb_lin = 2*k_turbin*Q_ref/(y_ref_normal**2 + epsilon_y)

A_lin = np.array([
    [-r_t_lin/M_tunnel, 0.0, -1.0/M_tunnel],
    [0.0, -(r_p_lin+r_turb_lin)/M_trykkrør, 1.0/M_trykkrør],
    [1.0/A_sjakt, -1.0/A_sjakt, 0.0]
])

print("A_lin =
", A_lin)

## Oppgave E1: Egenverdier

Beregn egenverdier og egenvektorer. For en kompleks egenverdi

$$\lambda=\alpha+i\omega$$

kan

- $\alpha$ tolkes som demping,
- $\omega$ som vinkelfrekvens,
- $2\pi/|\omega|$ som omtrentlig periode.

In [ ]:
egenverdier, P = ...
print("Egenverdier:", egenverdier)
print("Egenvektorer:
", P)

for lam in egenverdier:
    if abs(lam.imag) > 1e-10:
        print("Oscillasjonsperiode:", 2*np.pi/abs(lam.imag), "s")

## Oppgave E2: Direkte og modal løsning

Dersom $A_{lin}$ har tre lineært uavhengige egenvektorer, bruk

$$A_{lin}=PDP^{-1}$$

og variabelbyttet

$$\xi=Pz.$$

Sammenlign:

- Euler på det direkte lineære systemet,
- Euler på de frakoblede modalligningene,
- den ikke-lineære modellen ved en liten forstyrrelse.

In [ ]:
# Kontroller diagonaliserbarhet og implementer modal simulering.

## E.3 Parameterstudie

Undersøk hvordan egenverdiene endres når

- svingesjaktarealet dobles eller halveres,
- tunnellengden endres,
- tunnelarealet endres,
- friksjonen endres,
- trykkrøret blir lengre.

Forklar hvilke endringer som påvirker oscillasjonsperiode og demping.

# Fordypning: Generatorhastighet som femte tilstand

En mulig utvidelse er

$$
\boxed{M_\omega\dot{\Delta\omega}=P_m-P_e-D_\omega\Delta\omega.}
$$

Da blir tilstanden

$$x=(Q_t,Q_p,z,y,\Delta\omega)^T.$$

Dette kan brukes til å undersøke hvordan pumpekraftverket reagerer på en plutselig endring i elektrisk last. En full modell vil også kreve generator-, nett- og regulatormodeller og blir raskt langt større enn hovedprosjektet.

# Overgang til Matematikk 2: vannhammer som PDE

I Matematikk 1 representeres hvert rør med én samlet vannføring. For raske hendelser i lange trykkrør må trykkhøyde og vannføring avhenge av både posisjon og tid:

$$H=H(x,t),\qquad Q=Q(x,t).$$

En linearisert vannhammermodell kan skrives

$$
\boxed{
\frac{\partial H}{\partial t}
+
\frac{a^2}{gA}
\frac{\partial Q}{\partial x}
=0,}
$$

$$
\boxed{
\frac{\partial Q}{\partial t}
+
gA\frac{\partial H}{\partial x}
+
\text{friksjonsledd}
=0.}
$$

Her er $a$ trykkbølgens hastighet. Et passende referanseintervall er omtrent $1000$ til $1200$ m/s for de aktuelle vannvei-modellene.

En Matematikk 2-utvidelse kan:

- dele trykkrøret i romlige segmenter,
- bruke karakteristikkmetoden eller en annen romlig diskretisering,
- sammenligne langsom og rask ledeskovllukking,
- beregne trykkbølgens forplantning,
- sammenligne ODE- og PDE-modellen,
- undersøke hvor mange segmenter som trengs.

Dette gir progresjonen

$$
\boxed{
\text{Matematikk 1: samlet vannkolonne}
\longrightarrow
\text{Matematikk 2: trykk og vannføring langs røret}.}
$$

# Modellkritikk

Diskuter minst seks punkter:

- Del A bruker en lineær hydraulisk konduktans.
- Hoved-ODE-en bruker kvadratisk tap, men en svært enkel tapsmodell.
- Magasinnivåene holdes konstante.
- Svingesjakten har konstant tverrsnittsareal.
- Orifis, overløp og luftinnblanding er utelatt.
- Turbinens trykkfall beskrives med én enkel funksjon.
- Turbinvirkningsgraden er konstant.
- Pumpedrift beskrives ikke med en egen pumpekurve.
- Ledeskovlmodellen har ingen hastighets- eller posisjonsbegrensning.
- Generatorhastighet og elektrisk nett er utelatt fra hovedmodellen.
- Modellen beskriver ikke kavitasjon.
- Rørelastisitet og vannets kompressibilitet er utelatt.
- Euler krever kontroll av tidssteget.
- Den lineære egenverdianalysen gjelder bare nær referansetilstanden.
- Parameterne er representative undervisningsdata, ikke et komplett anleggsdesign.

## Mulige videreføringer

- separat pumpe- og turbinkarakteristikk,
- variable magasinshøyder,
- to aggregater med felles tunnel,
- orifis i svingesjakten,
- reliefventil,
- begrenset ledeskovlhastighet,
- generatorhastighet og frekvensregulering,
- optimal overgang mellom pumpe- og turbinedrift,
- PDE-modell for vannhammer,
- sammenligning med måledata.

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvordan forbindelsesmatrisen beskrev det stasjonære anlegget,
2. hvorfor nettverksmatrisen var singulær før magasinshøydene ble festet,
3. hvordan vannføringer og effekt ble kontrollert,
4. hvordan momentbalansen ga en skalar vannkolonne-ODE,
5. hvorfor $Q|Q|$ brukes i det ikke-lineære tapsleddet,
6. hvordan svingesjakten koblet tunnel- og trykkrørstrømningene,
7. hvordan lastøkning og lastavslag påvirket sjaktnivået,
8. hvordan ledeskovltregheten påvirket effektreguleringen,
9. hva egenverdiene fortalte om demping og oscillasjonsperiode,
10. hvorfor raske trykkbølger krever en PDE-modell.

## Faglig bakgrunn

Referanseparametrene er valgt med utgangspunkt i publiserte, representative hydrauliske turbinmodeller og deretter forenklet for Matematikk 1. Hovedmodellen beholder vannkolonnens treghet, svingesjaktens lagring og servotreghet, mens generator-, aksel-, strømnett- og detaljerte regulatormodeller er utelatt.